# CaRL — 8-shot vs 32-shot reachability

Ten notebook pokazuje **rzeczywiste zapisane obserwacje**, bez wywołań modeli, klucza ani `.env`. Celem jest exact hit stanu, łącznie z pozycją gracza. `k` to offset akcji w świadkowej trajektorii, a nie najkrótsza odległość; `L=32` jest osobnym limitem.

Otwórz plik i wybierz **Restart Kernel → Run All**. Bez konfiguracji notebook wybiera dostępny końcowy bundle, następnie najnowszy częściowy bundle, a gdy go nie ma — najdłuższy dostępny zweryfikowany prefix prawdziwego runu. Wybrana ścieżka i status są zawsze widoczne. Prefix pokazuje tylko wyniki ukończonych plansz, **nie pełne p(k) dla N=20**. Brak wybranego źródła powoduje błąd; żadne mocki nie są podstawiane.

Aby wybrać źródło ręcznie, ustaw `CARL_8V32_RESULTS` na katalog bundle albo katalog `paired_prefix_preview_*` przed uruchomieniem kernela. Bundle zawiera prywatne ślady ewaluacyjne i surowe odpowiedzi; nie przekazuj go policy modelu.

In [ ]:
import os, sys, json, hashlib, re, html
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, HTML

cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents, cwd / "CaRL"]
root = next((p for p in candidates if (p / "carl/icl/boards.py").is_file() and (p / "scripts/reachability_runner/independent_verify.py").is_file()), None)
explicit = os.environ.get("CARL_8V32_RESULTS")
if "CARL_8V32_RESULTS" in os.environ and not explicit:
    raise ValueError("CARL_8V32_RESULTS jest puste; wskaż katalog lub usuń zmienną.")
if explicit:
    source = Path(explicit).expanduser().resolve()
else:
    if root is None:
        raise RuntimeError("Nie znaleziono CaRL ani bundle. Uruchom kernel w repo/bundle lub ustaw CARL_8V32_RESULTS.")
    if (root / "results/snapshot.json").is_file():
        source = root
    else:
        run = root / "analysis/icl_reachability/figure1_8v32_exploratory_20_v1"
        final = run / "final_bundle"
        if (final / "results/snapshot.json").is_file():
            source = final
        else:
            partial_bundles = [path for path in run.glob("partial_bundle_*")
                               if (path / "results/snapshot.json").is_file()]
            if partial_bundles:
                # Partial snapshots are immutable; choose the most recently written snapshot.
                source = max(partial_bundles, key=lambda path: ((path / "results/snapshot.json").stat().st_mtime_ns, path.name))
            else:
                options = []
                for path in run.glob("paired_prefix_preview_*"):
                    match = re.search(r"paired_prefix_preview_(\d+)", path.name)
                    if match and (path / "preview_data.json").is_file():
                        options.append((int(match.group(1)), "verified" in path.name, path))
                if not options:
                    raise FileNotFoundError("Brak końcowego/częściowego bundle i zweryfikowanego prefixu w " + str(run))
                source = max(options, key=lambda value: (value[0], value[1], str(value[2])))[2]
if not source.is_dir():
    raise FileNotFoundError("Brak wskazanego katalogu wyników: " + str(source))
if root is None:
    root = next((p for p in (source, *source.parents) if (p / "carl/icl/boards.py").is_file()
                 and (p / "scripts/reachability_runner/independent_verify.py").is_file()), None)
if (source / "results/snapshot.json").is_file():
    mode = "bundle"
    code_root = source
elif (source / "preview_data.json").is_file():
    mode = "preview"
    code_root = root
else:
    raise FileNotFoundError("Brak snapshot.json lub preview_data.json w " + str(source) + "; nie podstawiono danych.")
if code_root is None or not (code_root / "carl/icl/boards.py").is_file():
    raise FileNotFoundError("Brak lokalnego kodu symulatora do replay.")
if mode == "bundle":
    checksums = source / "SHA256SUMS"
    if not checksums.is_file():
        raise FileNotFoundError("Bundle nie zawiera SHA256SUMS; nie uruchomiono replay.")
    entries = checksums.read_text(encoding="utf-8").splitlines()
    for entry in entries:
        digest, relative = entry.split("  ", 1)
        path = (source / relative).resolve()
        if not path.is_relative_to(source.resolve()) or not path.is_file():
            raise ValueError("Nieprawidłowa ścieżka w SHA256SUMS: " + relative)
        if hashlib.sha256(path.read_bytes()).hexdigest() != digest:
            raise ValueError("Hash bundle nie zgadza się: " + relative)
    print("SHA256SUMS PASS:", len(entries), "plików (manifest nie jest podpisem autentyczności).")
if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))
print("ŹRÓDŁO:", source)
print("TRYB:", mode.upper(), "— wcześniej zweryfikowane obserwacje LIVE; bez nowych requestów")

## Wczytanie pomiarów i statusów

Docelowo N=20 różnych layoutów na każdą kombinację `k × model × few-shot`, R=1. Prefix N<20 jest wyraźnie opisowy. Brakujące lub przerwane próby nie są porażkami. Każdy panel wykresu pokazuje wspólne plansze dla modeli i wariantów; powtórzenia stanów z jednego layoutu nie są dodatkowymi planszami.

In [ ]:
OFFSETS = (1, 2, 4, 8, 12, 16)
if mode == "preview":
    payload_path = source / "preview_data.json"
    doc = json.loads(payload_path.read_text(encoding="utf-8"))
    if doc.get("schema") != "paired-completed-prefix-preview-v2":
        raise ValueError("Nieznany schemat prefixu; nie użyto danych zastępczych.")
    records = doc["trials"]
    summary = pd.DataFrame(doc["summary"]).rename(columns={"shot": "shots", "layout_count": "resolved_layouts", "coverage_of_planned_cohort": "coverage"})
    summary["planned_layouts"] = 20
    summary["status"] = "PRELIMINARY"
    summary["p_hat"] = np.nan
    summary["resolved_rate_descriptive"] = summary["hits"] / summary["resolved"]
    summary["bootstrap_95_low"] = np.nan
    summary["bootstrap_95_high"] = np.nan
    print("PRELIMINARY:", doc["cohort_coverage"], "— obserwowane hits/resolved, bez CI; to nie jest p(k) N=20.")
else:
    meta = json.loads((source / "results/snapshot.json").read_text(encoding="utf-8"))
    if meta.get("schema") != "figure1-paired-replay-bundle-v1":
        raise ValueError("Nieznany schemat bundle.")
    payload_path = source / "results/plot_data.csv"
    summary = pd.read_csv(payload_path)
    records = []
    for line in (source / "results/trajectories.jsonl").read_text(encoding="utf-8").splitlines():
        saved = json.loads(line)
        identity, result = saved["identity"], saved["result"]
        records.append({**saved, "arm": identity["arm"], "k": identity["k"], "layout": identity["layout"],
                        "pair": identity["pair"], "trial_id": identity["key"], "seed": identity["seed"],
                        "exact_hit": result["exact_hit"] if saved["status"] == "complete" else False,
                        "first_hit": result["first_hit"], "stop_reason": result["stop_reason"]})
    resolved_count = meta["counts"].get("complete", 0) + meta["counts"].get("invalid_model_response", 0)
    bundle_status = "COMPLETE" if resolved_count == meta["planned"] else "PARTIAL"
    print("Bundle:", meta["classification"], "| STATUS", bundle_status,
          "| rozstrzygnięte", resolved_count, "/", meta["planned"], "| statusy", meta["counts"])
    print("Zweryfikowane ślady w bundle:", meta["trajectories"], "; bootstrap:", meta["bootstrap_unit"])
if len(summary) != 24 or {(int(x.shots), x.arm, int(x.k)) for x in summary.itertuples()} != {(s, a, k) for s in (8,32) for a in ("luna","terra_6_1") for k in OFFSETS}:
    raise ValueError("Eksport nie zawiera pełnej siatki 8/32 × model × k.")
if len({r["trial_id"] for r in records}) != len(records):
    raise ValueError("Powtórzony identyfikator triala.")
print("SHA-256 danych wykresu:", hashlib.sha256(payload_path.read_bytes()).hexdigest())
print("Liczba śladów:", len(records), "; K=", list(OFFSETS), "; L=32, R=1")
display(summary[["shots", "arm", "k", "planned_layouts", "resolved", "hits", "coverage", "status", "p_hat", "resolved_rate_descriptive"]])

## Niezależny replay zapisanych trajektorii

Każdy ślad jest ponownie odtwarzany dokładnym symulatorem od zapisanego source. Sprawdzane są wszystkie plansze, hashe przed i po akcjach oraz losowanie akcji z zapisanych rozkładów i seeda. Wynik replay nie tworzy nowej obserwacji modelu. Przy przerwanym trialu weryfikowany jest tylko zapisany prefix.

In [ ]:
import contextlib, io
from carl.icl.boards import from_rows, to_rows, state_hash
from scripts.reachability_runner.independent_verify import replay_actions, check_action_rng

# A preview needs the original frozen manifests: they authenticate its source,
# target, trial identity and seed even for a miss or a truncated rollout.
if mode == "preview":
    manifest_root = root / "analysis/icl_reachability/figure1_8v32_exploratory_20_v1"
    frozen_jobs = {}
    for shot in (8, 32):
        manifest = json.loads((manifest_root / f"study_{shot}/manifest.json").read_text(encoding="utf-8"))
        frozen_jobs[shot] = {job["key"]: job for job in manifest["jobs"]}

for record in records:
    job = record["identity"] if mode == "bundle" else frozen_jobs[int(record["shot"])].get(record["trial_id"])
    if not job or any(record[field] != job[field if field != "trial_id" else "key"] for field in ("trial_id", "arm", "k", "layout", "pair", "seed")):
        raise ValueError("Trial nie zgadza się z zamrożonym manifestem: " + record["trial_id"])
    start, target = from_rows(record["source"]), from_rows(record["target"])
    if state_hash(start) != job["source_hash"] or state_hash(target) != job["target_hash"]:
        raise ValueError("Hash source/target różni się od manifestu: " + record["trial_id"])
    with contextlib.redirect_stdout(io.StringIO()):
        replay = replay_actions(start, target, record["steps"], 32)
    check_action_rng(record["steps"], int(record["seed"]))
    if [to_rows(state) for state in replay["states"]] != record["frames"]:
        raise ValueError("Zapisane plansze różnią się od symulatora: " + record["trial_id"])
    if record["status"] == "complete" and (replay["exact_hit"] != record["exact_hit"] or replay["first_hit"] != record["first_hit"] or replay["stop_reason"] != record["stop_reason"]):
        raise ValueError("Wynik triala różni się od replay: " + record["trial_id"])
print("REPLAY PASS:", len(records), "zapisanych śladów; source/target, akcje, RNG i każda klatka")

## Reachability względem offsetu `k`

Dla ukończonej komórki p(k) = exact hits / 20 z równą wagą layoutów. Na niepełnym prefixie pokazujemy wyłącznie opisowe hits/resolved; nie jest to estymata pełnej kohorty. Przedziały 95% są rysowane tylko jeśli zostały obliczone dla kompletnej kohorty bootstrapem całych layoutów. Brak trafień nie dowodzi, że prawdziwe p=0.

In [ ]:
# Recompute counts from independently replayed traces rather than trusting the plot table.
for row in summary.itertuples():
    selected = [r for r in records if int(r["shot"]) == int(row.shots) and r["arm"] == row.arm and int(r["k"]) == int(row.k)]
    resolved = [r for r in selected if r["status"] in {"complete", "invalid_model_response"}]
    hits = sum(r["status"] == "complete" and r["exact_hit"] is True for r in resolved)
    if len(resolved) != int(row.resolved) or hits != int(row.hits):
        raise ValueError("Mianownik lub liczba trafień nie zgadza się z replay.")
    if pd.notna(row.p_hat) and (int(row.resolved) != int(row.planned_layouts) or not np.isclose(row.p_hat, hits / int(row.planned_layouts))):
        raise ValueError("Oficjalne p(k) nie ma kompletnego mianownika.")

partial = mode == "preview" or (summary["resolved"] < summary["planned_layouts"]).any()
fig, ax = plt.subplots(figsize=(9, 5.5))
for shot in (8, 32):
    for arm, color in (("luna", "tab:blue"), ("terra_6_1", "tab:orange")):
        cell = summary[(summary.shots == shot) & (summary.arm == arm)].set_index("k").reindex(OFFSETS)
        if partial:
            values = pd.to_numeric(cell["resolved_rate_descriptive"], errors="coerce")
        else:
            values = pd.to_numeric(cell["p_hat"], errors="coerce")
        ax.plot(OFFSETS, values, marker="o", color=color, linestyle="-" if shot == 8 else "--", label=f"{arm}, {shot}-shot")
        if not partial:
            for k, row in cell.iterrows():
                if pd.notna(row["bootstrap_95_low"]) and pd.notna(row["bootstrap_95_high"]):
                    ax.vlines(k, row["bootstrap_95_low"], row["bootstrap_95_high"], color=color, alpha=.45)
ax.set(xlabel="Offset k (liczba akcji świadka)", ylabel="Exact hits / resolved" if partial else "Exact-target reachability p(k)",
       title="PARTIAL — opisowe hits/resolved; nie p(k) całej kohorty" if partial else "Reachability — 20 layoutów, R=1",
       xticks=OFFSETS, ylim=(-.04, 1.04))
ax.grid(alpha=.22); ax.legend(); fig.tight_layout(); plt.show()
print("Źródło wykresu:", payload_path, "; coverage w tabeli powyżej.")
if partial:
    print("PARTIAL: dane opisowe; brakujące próby nie są porażkami i nie wyznaczono CI.")

## Przykładowe pełne ślady

Poniżej pokazujemy deterministycznie po jednym trafieniu i nietrafieniu, jeśli są dostępne. Przewiń poziomo, aby zobaczyć wszystkie klatki, akcje i hashe. Cały zbiór śladów i requestów pozostaje w pliku źródłowym. Przykłady ilustracyjne nie są dodatkowymi próbami.

In [ ]:
ACTION_NAMES = ("UP", "DOWN", "LEFT", "RIGHT")
for title, matches in (("Exact hit", [r for r in records if r["status"] == "complete" and r["exact_hit"]]),
                       ("Miss", [r for r in records if r["status"] == "complete" and not r["exact_hit"]])):
    if not matches:
        print(title + ": brak dostępnego śladu")
        continue
    item = sorted(matches, key=lambda r: r["trial_id"])[0]
    print(title, "| trial", item["trial_id"], "|", item["arm"], f"{item['shot']}-shot", "| k", item["k"], "| first_hit", item["first_hit"], "| stop", item["stop_reason"])
    cards = []
    for t, frame in enumerate(item["frames"]):
        action = "start" if t == 0 else ACTION_NAMES[item["steps"][t-1]["action"]]
        digest = state_hash(from_rows(frame))
        cards.append("<div style='flex:0 0 auto;padding:8px;border:1px solid #aaa'><b>t=" + str(t) + " · " + action + "</b><br><small>" + digest + "</small><pre>" + html.escape("\n".join(frame)) + "</pre></div>")
    display(HTML("<div style='display:flex;gap:8px;overflow-x:auto'>" + "".join(cards) + "</div>"))

Dla powtarzalności zachowaj cały prywatny bundle: manifesty, stany, cache/odpowiedzi i ledgery. Replay tych zapisanych odpowiedzi jest deterministyczny; ponowne wywołanie API może zwrócić inne rozkłady.